In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

ModuleNotFoundError: No module named 'pandas'

In [2]:
# 1. Carga de datos
archivo_csv = "datos_procesados.csv"  # Asegúrate de que el archivo está en la ruta correcta
df = pd.read_csv(r"/Users/alejandrabenavidessanclemente/Desktop/ProyectoGradoGEB/ProyectoGradoGEB/datos_procesados.csv")

In [3]:
# 2. Definir variable objetivo
target = 'ORIG_RAW_VOLUME'

In [6]:
 3. Eliminar Variables no necesarias con la Correlación
# Filtrar solo variables numéricas
X_numeric = df.select_dtypes(include=[np.number])

# Calcular la correlación con la variable objetivo
corr_with_target = X_numeric.corr()[target]

# Definir umbrales para eliminar variables redundantes
upper_threshold = 0.98  # Correlación muy alta → redundante
lower_threshold = 0.05   # Correlación muy baja → no aporta información

# Encontrar las variables a eliminar
columns_to_drop = corr_with_target[(corr_with_target.abs() > upper_threshold) | (corr_with_target.abs() < lower_threshold)].index.tolist()

# No eliminar la variable objetivo
if target in columns_to_drop:
    columns_to_drop.remove(target)

# Filtrar solo las columnas que realmente existen en X antes de eliminarlas
columns_to_drop = [col for col in columns_to_drop if col in df.columns]

# Eliminar las columnas redundantes
df = df.drop(columns=columns_to_drop)

print("Columnas eliminadas:", columns_to_drop)
print("Columnas restantes:", df.columns)

Correlación entre variables:
                   ORIG_STD_VOLUME  STD_VOLUME  ORIG_TEMPERATURE  TEMPERATURE  \
ORIG_STD_VOLUME          1.000000    0.999857          0.336591     0.336591   
STD_VOLUME               0.999857    1.000000          0.336702     0.336702   
ORIG_TEMPERATURE         0.336591    0.336702          1.000000     1.000000   
TEMPERATURE              0.336591    0.336702          1.000000     1.000000   
PRESSURE                 0.232915    0.232641          0.128581     0.128581   
ORIG_PRESSURE            0.232915    0.232641          0.128581     0.128581   
RAW_VOLUME               0.360182    0.360895          0.034216     0.034216   

                  PRESSURE  ORIG_PRESSURE  RAW_VOLUME  
ORIG_STD_VOLUME   0.232915       0.232915    0.360182  
STD_VOLUME        0.232641       0.232641    0.360895  
ORIG_TEMPERATURE  0.128581       0.128581    0.034216  
TEMPERATURE       0.128581       0.128581    0.034216  
PRESSURE          1.000000       1.000000   -0.73

KeyError: "['ORIG_RAW_VOLUME'] not found in axis"

In [ ]:
# 4. Normalizar los datos
# Definir X e Y
X = df.drop(columns=[target])
y = df[target]

# a. Asegurar que X no contenga columnas de tipo datetime
X = df.drop(columns=[target])  # Eliminar la variable objetivo

# b. Verificar si hay columnas de tipo datetime y eliminarlas
X = X.select_dtypes(exclude=['datetime', 'object'])

#Aplicar escalado estándar solo a variables numéricas
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Escalado realizado correctamente.")

In [ ]:
# 5. Selección de Variables Automática (RFE)
# Usamos una regresión lineal como base para RFE
model_rfe = LinearRegression()
selector = RFE(model_rfe, n_features_to_select=5)  # Seleccionar las 5 mejores variables
X_selected = selector.fit_transform(X_scaled, y)

# Obtener nombres de las variables seleccionadas
selected_columns = X.columns[selector.support_]
print(f"Variables seleccionadas para predecir ORIG_RAW_VOLUME: {selected_columns}")

In [ ]:
# 6. Encontrar los mejores Hiperparámetros
# División en entrenamiento y prueba
train_size = int(len(df) * 0.7)
X_train, X_test = X_selected[:train_size], X_selected[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Búsqueda automática de Hiperparámetros
auto_model = auto_arima(y_train, exogenous=X_train,
                        seasonal=True, m=24,  # Estacionalidad horaria
                        stepwise=True, 
                        start_p=0, max_p=2,  
                        start_q=1, max_q=2, 
                        max_P=1, max_Q=1,  
                        max_d=1, max_D=1, 
                        trace=True, suppress_warnings=True,
                        n_jobs=1) x